In [ ]:
"""
Project 08: Viterbi decoding with a hidden Markov model (HMM).

The HMM class stores the three probability tables (initial, transition,
emission), checks them, converts them to logs once, and runs Viterbi.
"""

import math

NEG_INF = float("-inf")
TOLERANCE = 1e-6


class HMM:
    # PART 1: build the model

    def __init__(self, init_probs, trans_probs, emit_probs):
        # Save the original tables
        self.init_probs = init_probs
        self.trans_probs = trans_probs
        self.emit_probs = emit_probs

        # Get states and symbols from the data (never hard-coded)
        # example: states = ['I', 'G'], symbols = ['A', 'C', 'G', 'T']
        self.states = list(init_probs.keys())
        if not self.states:
            raise ValueError("init_probs has no states")
        first_state = self.states[0]
        self.symbols = list(emit_probs.get(first_state, {}).keys())

        # Check the tables, then convert everything to logs once
        self._validate()
        self._convert_to_logs()

    @staticmethod
    def safe_log(prob):
        """Natural log, except log(0) = -inf (an impossible event)."""
        if prob == 0:
            return NEG_INF
        return math.log(prob)

    def _validate(self):
        states = set(self.states)
        symbols = set(self.symbols)

        # 1. Keys must match across the three tables
        if set(self.trans_probs.keys()) != states:
            raise ValueError("trans_probs states don't match init_probs states")
        if set(self.emit_probs.keys()) != states:
            raise ValueError("emit_probs states don't match init_probs states")
        for s in self.states:
            if set(self.trans_probs[s].keys()) != states:
                raise ValueError(f"trans_probs['{s}'] doesn't list every state")
            if set(self.emit_probs[s].keys()) != symbols:
                raise ValueError(f"emit_probs['{s}'] doesn't list every symbol")

        # 2. Every value must be a real probability
        all_values = list(self.init_probs.values())
        for s in self.states:
            all_values += list(self.trans_probs[s].values())
            all_values += list(self.emit_probs[s].values())
        for p in all_values:
            if not 0 <= p <= 1:
                raise ValueError(f"probability {p} is not between 0 and 1")

        # 3. Every row must add up to 1 (with a little rounding room)
        if abs(sum(self.init_probs.values()) - 1) > TOLERANCE:
            raise ValueError("init_probs don't add up to 1")
        for s in self.states:
            if abs(sum(self.trans_probs[s].values()) - 1) > TOLERANCE:
                raise ValueError(f"trans_probs['{s}'] don't add up to 1")
            if abs(sum(self.emit_probs[s].values()) - 1) > TOLERANCE:
                raise ValueError(f"emit_probs['{s}'] don't add up to 1")

    def _convert_to_logs(self):
        # example: log_init['G'] = ln(0.8) = -0.2231
        self.log_init = {s: self.safe_log(self.init_probs[s]) for s in self.states}
        # example: log_trans['G']['G'] = ln(0.9) = -0.1054
        self.log_trans = {
            i: {j: self.safe_log(self.trans_probs[i][j]) for j in self.states}
            for i in self.states
        }
        # example: log_emit['I']['C'] = ln(0.4) = -0.9163
        self.log_emit = {
            s: {x: self.safe_log(self.emit_probs[s][x]) for x in self.symbols}
            for s in self.states
        }

    # PART 2: Viterbi

    def viterbi(self, sequence):
        """
        Return (path, best_log_prob): the most likely hidden-state path for
        the sequence, and the natural log of that path's probability.
        """
        # Check the input
        if len(sequence) == 0:
            raise ValueError("empty sequence")
        for symbol in sequence:
            if symbol not in self.log_emit[self.states[0]]:
                raise ValueError(f"unknown symbol '{symbol}'")

        n = len(sequence)
        # score[t][state] = best log score for reaching this state at column t
        # arrow[t][state] = which state that best score came from
        score = [dict() for _ in range(n)]
        arrow = [dict() for _ in range(n)]

        # Column 1 (index 0): log initial + log emission
        # example: score[0]['G'] = ln(0.8) + ln(0.2) = -1.8326
        first = sequence[0]
        for s in self.states:
            score[0][s] = self.log_init[s] + self.log_emit[s][first]
            arrow[0][s] = None

        # Columns 2 to the end: try every way in, keep the best
        for t in range(1, n):
            symbol = sequence[t]
            for state_now in self.states:
                best_road = NEG_INF
                best_from = None
                for state_before in self.states:
                    road = score[t - 1][state_before] + self.log_trans[state_before][state_now]
                    if road > best_road:
                        best_road = road
                        best_from = state_before
                score[t][state_now] = best_road + self.log_emit[state_now][symbol]
                arrow[t][state_now] = best_from

        # Best ending: biggest score in the last column
        best_last_state = max(self.states, key=lambda s: score[n - 1][s])
        best_log_prob = score[n - 1][best_last_state]
        if best_log_prob == NEG_INF:
            raise ValueError("no possible path for this sequence")

        # Trace back: stand on a box, read its arrow, jump left
        path = []
        standing_on = best_last_state
        for t in range(n - 1, -1, -1):
            path.append(standing_on)
            standing_on = arrow[t][standing_on]
        path.reverse()

        return path, best_log_prob


# TESTS

def run_tests():
    init_probs = {"I": 0.2, "G": 0.8}
    trans_probs = {"I": {"I": 0.7, "G": 0.3}, "G": {"I": 0.1, "G": 0.9}}
    emit_probs = {
        "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
        "G": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3},
    }
    model = HMM(init_probs, trans_probs, emit_probs)

    # One symbol: G wins with 0.8 x 0.2 = 0.16
    path, lp = model.viterbi("G")
    assert path == ["G"] and math.isclose(lp, math.log(0.16)), (path, lp)

    # Two symbols: G G with 0.0288
    path, lp = model.viterbi("GG")
    assert path == ["G", "G"] and math.isclose(lp, math.log(0.0288)), (path, lp)

    # Three symbols: I I I with 0.006272 (worked by hand)
    path, lp = model.viterbi("GGC")
    assert path == ["I", "I", "I"] and math.isclose(lp, math.log(0.006272)), (path, lp)

    # The class example: all background
    path, lp = model.viterbi("GGCACTGAA")
    assert path == ["G"] * 9, path
    assert abs(lp - (-13.9291)) < 1e-3, lp

    # A different model: proves nothing is hard-coded
    primer = HMM(
        {"T": 0.2, "U": 0.8},
        {"T": {"T": 0.9, "U": 0.1}, "U": {"T": 0.05, "U": 0.95}},
        {"T": {"none": 0.05, "few": 0.15, "many": 0.80},
         "U": {"none": 0.70, "few": 0.28, "many": 0.02}},
    )
    obs = ["none", "few", "many", "many", "none", "many", "none"]
    path, lp = primer.viterbi(obs)
    assert path == ["U", "U", "T", "T", "T", "T", "U"], path
    assert abs(math.exp(lp) - 9.7e-6) < 0.1e-6, math.exp(lp)

    # Unknown symbol
    try:
        model.viterbi("GGNA")
        assert False, "should have raised"
    except ValueError:
        pass

    # Empty sequence
    try:
        model.viterbi("")
        assert False, "should have raised"
    except ValueError:
        pass

    # Broken table: emissions add up to 0.9
    try:
        HMM(init_probs, trans_probs,
            {"I": {"A": 0.1, "C": 0.4, "G": 0.3, "T": 0.1},
             "G": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3}})
        assert False, "should have raised"
    except ValueError:
        pass

    # Long sequence: logs keep the score a real number
    path, lp = model.viterbi("GGCACTGAA" * 200)
    assert len(path) == 1800 and lp != NEG_INF and not math.isnan(lp), lp

    print("All tests passed.")


if __name__ == "__main__":
    run_tests()

    model = HMM(
        {"I": 0.2, "G": 0.8},
        {"I": {"I": 0.7, "G": 0.3}, "G": {"I": 0.1, "G": 0.9}},
        {"I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
         "G": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3}},
    )
    path, best_log_prob = model.viterbi("GGCACTGAA")
    print("Path:          ", "".join(path))
    print("Log probability:", round(best_log_prob, 4))
    print("Probability:    ", f"{math.exp(best_log_prob):.3e}")

